##**Fake News Classifier Using LSTM**
Dataset: https://www.kaggle.com/c/fake-news/data#

In [1]:
import pandas as pd

In [2]:
news=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/train.txt', on_bad_lines='skip', engine='python')
news.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [3]:
news.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20822 entries, 0 to 20821
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      20822 non-null  object
 1   title   20257 non-null  object
 2   author  18847 non-null  object
 3   text    20763 non-null  object
 4   label   20800 non-null  object
dtypes: object(5)
memory usage: 813.5+ KB


In [4]:
news.isnull().sum()

,0
id,0
title,565
author,1975
text,59
label,22


In [5]:
###Drop Nan Values
news=news.dropna()

In [6]:
# Keep only rows where label is '0' or '1'
news = news[news['label'].isin(['0', '1'])]

# Now convert label to integer
y = news['label'].astype(int)


In [7]:
## Get the Independent Features

X=news.drop('label',axis=1)

In [8]:
print(news['label'].unique())

['1' '0']


In [9]:
## Get the Dependent features
y = news['label'].astype('int')  # or use 'float' if needed

In [10]:
X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18283 entries, 0 to 20821
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      18283 non-null  object
 1   title   18283 non-null  object
 2   author  18283 non-null  object
 3   text    18283 non-null  object
dtypes: object(4)
memory usage: 714.2+ KB


In [11]:
y.info()

<class 'pandas.core.series.Series'>
Index: 18283 entries, 0 to 20821
Series name: label
Non-Null Count  Dtype
--------------  -----
18283 non-null  int64
dtypes: int64(1)
memory usage: 285.7 KB


In [12]:
X.shape

(18283, 4)

In [13]:
y.shape

(18283,)

In [14]:
import tensorflow as tf

In [15]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense

In [16]:
### Vocabulary size
voc_size=5000

###**Onehot Representation**

In [17]:
messages=X.copy()

In [18]:
messages['title'][15]

'In Major League Soccer, Argentines Find a Home and Success - The New York Times'

In [19]:
messages.reset_index(inplace=True)

In [20]:
import nltk
import re
from nltk.corpus import stopwords

In [21]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [22]:
### Dataset Preprocessing
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
corpus = []
for i in range(0, len(messages)):
    print(i)
    review = re.sub('[^a-zA-Z]', ' ', messages['title'][i])  # Regular Expression
    review = review.lower()
    review = review.split()

    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)

Streaming output truncated to the last 5000 lines.
13283
13284
13285
13286
13287
13288
13289
13290
13291
13292
13293
13294
13295
13296
13297
13298
13299
13300
13301
13302
13303
13304
13305
13306
13307
13308
13309
13310
13311
13312
13313
13314
13315
13316
13317
13318
13319
13320
13321
13322
13323
13324
13325
13326
13327
13328
13329
13330
13331
13332
13333
13334
13335
13336
13337
13338
13339
13340
13341
13342
13343
13344
13345
13346
13347
13348
13349
13350
13351
13352
13353
13354
13355
13356
13357
13358
13359
13360
13361
13362
13363
13364
13365
13366
13367
13368
13369
13370
13371
13372
13373
13374
13375
13376
13377
13378
13379
13380
13381
13382
13383
13384
13385
13386
13387
13388
13389
13390
13391
13392
13393
13394
13395
13396
13397
13398
13399
13400
13401
13402
13403
13404
13405
13406
13407
13408
13409
13410
13411
13412
13413
13414
13415
13416
13417
13418
13419
13420
13421
13422
13423
13424
13425
13426
13427
13428
13429
13430
13431
13432
13433
13434
13435
13436
13437
13438
13439
13440
1

In [23]:
corpus

['hous dem aid even see comey letter jason chaffetz tweet',
 'flynn hillari clinton big woman campu breitbart',
 'truth might get fire',
 'civilian kill singl us airstrik identifi',
 'iranian woman jail fiction unpublish stori woman stone death adulteri',
 'jacki mason hollywood would love trump bomb north korea lack tran bathroom exclus video breitbart',
 'beno hamon win french socialist parti presidenti nomin new york time',
 'back channel plan ukrain russia courtesi trump associ new york time',
 'obama organ action partner soro link indivis disrupt trump agenda',
 'bbc comedi sketch real housew isi caus outrag',
 'russian research discov secret nazi militari base treasur hunter arctic photo',
 'us offici see link trump russia',
 'ye paid govern troll social media blog forum websit',
 'major leagu soccer argentin find home success new york time',
 'well fargo chief abruptli step new york time',
 'anonym donor pay million releas everyon arrest dakota access pipelin',
 'fbi close hilla

In [24]:
onehot_repr=[one_hot(words,voc_size)for words in corpus]
onehot_repr

[[3031, 167, 3408, 2295, 1099, 476, 3144, 948, 4093, 3483],
 [4354, 2489, 4339, 4994, 760, 2216, 4034],
 [2310, 1624, 1547, 4439],
 [2713, 4989, 546, 4946, 4964, 3935],
 [1468, 760, 356, 1512, 4563, 3618, 760, 1943, 4680, 934],
 [1092,
  1619,
  2734,
  1569,
  4411,
  3431,
  4847,
  2353,
  2901,
  729,
  4871,
  3612,
  418,
  1574,
  4034],
 [4983, 2680, 1306, 2053, 3559, 995, 2915, 4016, 681, 1595, 906],
 [4744, 2126, 2994, 2446, 3501, 4410, 3431, 2281, 681, 1595, 906],
 [1787, 2632, 1194, 2042, 3070, 260, 4209, 78, 3431, 1400],
 [2316, 4433, 4225, 2642, 1828, 3435, 1976, 2331],
 [3425, 4294, 793, 3804, 2911, 3326, 4870, 4946, 996, 1906, 3814],
 [4946, 3465, 1099, 260, 3431, 3501],
 [4597, 1493, 4772, 1101, 2469, 2137, 4436, 3840, 2286],
 [2644, 2858, 4609, 128, 1754, 1377, 1901, 681, 1595, 906],
 [4337, 3416, 3544, 538, 1975, 681, 1595, 906],
 [4195, 1303, 4246, 3149, 2656, 3458, 3070, 4352, 4010, 2257],
 [1497, 812, 2489],
 [1288, 3339, 3871, 1971, 3431, 675, 3079, 4034],
 [1300

###**Embedding Representation**

In [25]:
sent_length=20
embedded_docs=pad_sequences(onehot_repr,padding='pre',maxlen=sent_length)
print(embedded_docs)

[[   0    0    0 ...  948 4093 3483]
 [   0    0    0 ...  760 2216 4034]
 [   0    0    0 ... 1624 1547 4439]
 ...
 [   0    0    0 ...  681 1595  906]
 [   0    0    0 ... 1490 3641 2359]
 [   0    0    0 ... 3184 4295 3273]]


In [26]:
embedded_docs[0]

array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0, 3031,
        167, 3408, 2295, 1099,  476, 3144,  948, 4093, 3483], dtype=int32)

In [27]:
## Creating model
embedding_vector_features=40
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_length=sent_length))
model.add(LSTM(100))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [28]:
len(embedded_docs),y.shape

(18283, (18283,))

In [29]:
import numpy as np
X_final=np.array(embedded_docs)
y_final=np.array(y)

In [30]:
X_final.shape,y_final.shape

((18283, 20), (18283,))

In [31]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.33, random_state=42)

###**Model Training**

In [32]:
### Finally Training
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=64)

Epoch 1/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.7804 - loss: 0.4244 - val_accuracy: 0.9042 - val_loss: 0.2154
Epoch 2/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.9395 - loss: 0.1499 - val_accuracy: 0.9160 - val_loss: 0.1978
Epoch 3/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.9585 - loss: 0.1054 - val_accuracy: 0.9158 - val_loss: 0.2390
Epoch 4/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.9731 - loss: 0.0733 - val_accuracy: 0.9133 - val_loss: 0.2266
Epoch 5/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 11s 39ms/step - accuracy: 0.9797 - loss: 0.0658 - val_accuracy: 0.8951 - val_loss: 0.2905
Epoch 6/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - accuracy: 0.9850 - loss: 0.0511 - val_accuracy: 0.9062 - val_loss: 0.3051
Epoch 7/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 7s 34ms/step - accuracy: 0.9884 - loss: 0.0361 - val_accuracy: 0.9090 - val_loss: 0.3823
Epoch 8/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 10s 34ms/step - accuracy: 0.9913 - loss: 0.0308 - val_ac

###**Adding Dropout**

In [33]:
from tensorflow.keras.layers import Dropout
## Creating model
embedding_vector_features=40
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_length=sent_length))
model.add(Dropout(0.3))
model.add(LSTM(100))
model.add(Dropout(0.3))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

###**Performance Metrics And Accuracy**

In [34]:
y_pred = model.predict(X_test)

189/189 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step


In [35]:
# Use the predict method to get the probability outputs
y_pred_prob = model.predict(X_test)

189/189 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


In [36]:
# Convert the probabilities to class labels using a threshold (0.5 for sigmoid)
y_pred = (y_pred_prob > 0.5).astype(int)

In [37]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test,y_pred)

array([[2389, 1026],
       [ 904, 1715]])

In [38]:
from sklearn.metrics import accuracy_score

accuracy_score(y_test,y_pred)

0.6801458402386477

In [39]:
from sklearn.metrics import classification_report

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.73      0.70      0.71      3415
           1       0.63      0.65      0.64      2619

    accuracy                           0.68      6034
   macro avg       0.68      0.68      0.68      6034
weighted avg       0.68      0.68      0.68      6034



**Note:**

**Though the dataset is corrupted, that's why I am having bad prediction**